# 🐍 Employee Data Analysis & Visualization Pipeline
### NTUC LearningHub — Python Trainer Portfolio
**Domain:** Data Analytics, AI & Machine Learning  
**Tools:** pandas · NumPy · matplotlib · seaborn · OpenAI API

---
## 📋 Notebook Outline
1. Setup & Imports  
2. Data Ingestion  
3. Data Quality Report  
4. Data Cleaning & Transformation  
5. Export Clean Data  
6. Descriptive Analytics  
7. Data Visualizations (9 charts)  
8. AI-Assisted Insights (Optional — requires OpenAI API Key)


## 1️⃣ Setup & Imports

In [ ]:
%matplotlib inline
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.05)
Path("output_charts").mkdir(exist_ok=True)

# Optional AI integration
try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

print("✅ All imports successful")
print(f"pandas {pd.__version__} | numpy {np.__version__}")

## 2️⃣ Data Ingestion
Load the raw CSV and inspect its shape, columns and first few rows.

In [ ]:
RAW_FILE   = "raw_data.csv"
CLEAN_FILE = "clean_data.csv"

df_raw = pd.read_csv(RAW_FILE)
df_raw.columns = df_raw.columns.str.strip()   # strip column name whitespace

print(f"Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns\n")
print("Columns:", list(df_raw.columns))
df_raw.head(5)

In [ ]:
# Data types overview
df_raw.dtypes

## 3️⃣ Data Quality Report
Before cleaning, we audit the dataset for issues that need to be resolved.

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
quality_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct, "Dtype": df_raw.dtypes})
print("── Missing Values ──")
display(quality_df[quality_df["Missing Count"] > 0])

In [ ]:
# Anomaly checks
print("Duplicates:", df_raw.duplicated().sum())

# Negative salaries
sal = pd.to_numeric(df_raw["Salary"].astype(str).str.strip(), errors="coerce")
print("Negative Salaries:", (sal < 0).sum(), "→", sal[sal < 0].tolist())

# Unrealistic ages
age = pd.to_numeric(df_raw["Age"], errors="coerce")
print("Unrealistic Ages (>100 or <18):", ((age > 100) | (age < 18)).sum())

# Inconsistent categoricals
print("Gender  values:", df_raw["Gender"].unique())
print("Status  values:", df_raw["Status"].unique())

## 4️⃣ Data Cleaning & Transformation
### Issues Fixed:
| Problem | Solution |
|---------|----------|
| Mixed date formats | Multi-format parser |
| Impossible date (30-Feb) | → NaT |
| Negative salary | → NaN → median imputation |
| Age = 200 | → NaN → median imputation |
| Inconsistent gender (M / F / Male / Female) | Normalised map |
| Mixed-case Status (active / ACTIVE) | `.str.title()` |
| Trailing whitespace in column names | `.str.strip()` |


In [ ]:
def try_parse_date(d):
    if pd.isnull(d) or str(d).strip() in ("", "nan"):
        return pd.NaT
    for fmt in ["%d-%b-%Y", "%Y-%m-%d", "%Y/%m/%d", "%m/%d/%Y", "%d/%m/%Y"]:
        try:
            return datetime.strptime(str(d).strip(), fmt)
        except ValueError:
            continue
    return pd.NaT

def clean(df):
    df = df.copy()

    # Strip whitespace from all string columns
    for col in df.select_dtypes("object").columns:
        df[col] = df[col].astype(str).str.strip()
    df.replace({"nan": np.nan, "": np.nan, "None": np.nan}, inplace=True)

    # Fix Salary
    df["Salary"] = pd.to_numeric(
        df["Salary"].astype(str).str.replace(",","").str.replace(" ",""), errors="coerce")
    df.loc[df["Salary"] < 0, "Salary"] = np.nan

    # Fix Age
    df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
    df.loc[(df["Age"] > 100) | (df["Age"] < 16), "Age"] = np.nan

    # Fix Performance Score & Bonus
    df["Performance Score"] = pd.to_numeric(df["Performance Score"], errors="coerce")
    df["Monthly Bonus"]     = pd.to_numeric(df["Monthly Bonus"], errors="coerce")

    # Normalise Gender & Status
    df["Gender"] = df["Gender"].map({"Male":"Male","male":"Male","M":"Male",
                                     "Female":"Female","female":"Female","F":"Female"})
    df["Status"]    = df["Status"].str.title()
    df["Full Name"] = df["Full Name"].str.title()
    df.loc[df["Full Name"].str.strip() == "Nan", "Full Name"] = np.nan

    # Parse dates
    df["Join Date"] = df["Join Date"].apply(try_parse_date)

    # Median imputation
    for col in ["Age", "Salary", "Performance Score", "Monthly Bonus"]:
        df[col].fillna(df[col].median(), inplace=True)

    # Drop rows without Employee ID or duplicate IDs
    df.dropna(subset=["Employee ID"], inplace=True)
    df.drop_duplicates(subset=["Employee ID"], inplace=True)

    # Feature Engineering
    df["Tenure Years"] = df["Join Date"].apply(
        lambda d: round((datetime.now()-d).days/365.25, 1) if pd.notna(d) else np.nan)

    def seniority(y):
        return "Junior" if y<=2 else "Mid-Level" if y<=7 else "Senior" if y<=15 else "Expert"
    def sal_band(s):
        return "Entry (<5k)" if s<5000 else "Mid (5k-8k)" if s<8000 else "Senior (8k-11k)" if s<11000 else "Leadership (>11k)"

    df["Seniority Band"] = df["Years Experience"].apply(seniority)
    df["Salary Band"]    = df["Salary"].apply(sal_band)
    df["Bonus Ratio %"]  = ((df["Monthly Bonus"] / df["Salary"]) * 100).round(2)

    df.reset_index(drop=True, inplace=True)
    return df

df = clean(df_raw)
print(f"✅ Clean dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 5️⃣ Export Clean Data

In [ ]:
df.to_csv(CLEAN_FILE, index=False, date_format="%Y-%m-%d")
print(f"✅ Saved → {CLEAN_FILE}")
print(f"   Shape : {df.shape}")

## 6️⃣ Descriptive Analytics

In [ ]:
# Numeric summary
df[["Age","Salary","Performance Score","Years Experience",
    "Monthly Bonus","Tenure Years","Bonus Ratio %"]].describe().round(2)

In [ ]:
# Headcount & avg salary by department
dept_stats = df.groupby("Department").agg(
    Headcount=("Employee ID","count"),
    Avg_Salary=("Salary","mean"),
    Median_Salary=("Salary","median"),
    Avg_Perf=("Performance Score","mean")
).round(1).sort_values("Avg_Salary", ascending=False)
display(dept_stats)

In [ ]:
# Gender distribution
print(df["Gender"].value_counts())
print()
# Correlation matrix
df[["Age","Salary","Years Experience","Performance Score","Monthly Bonus"]].corr().round(3)

In [ ]:
# Top 5 performers
df[["Full Name","Department","Salary","Performance Score","Seniority Band"]] \
  .sort_values("Performance Score", ascending=False).head(5).reset_index(drop=True)

## 7️⃣ Data Visualizations

In [ ]:
# Chart 1 — Salary Distribution by Department
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x="Department", y="Salary", palette="Set2", showfliers=False, width=0.55, ax=ax)
sns.stripplot(data=df, x="Department", y="Salary", color="#444", alpha=0.45, jitter=True, size=5, ax=ax)
ax.set_title("Salary Distribution by Department", fontsize=15, fontweight="bold")
ax.set_ylabel("Monthly Salary (SGD)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))
plt.tight_layout(); plt.show()

In [ ]:
# Chart 2 — Avg Salary vs Headcount (Dual-axis)
dept_sum = df.groupby("Department").agg(Headcount=("Employee ID","count"), Avg=("Salary","mean")).sort_values("Avg", ascending=False)
fig, ax1 = plt.subplots(figsize=(11, 6))
colors = sns.color_palette("Set2", len(dept_sum))
ax1.bar(dept_sum.index, dept_sum["Avg"], color=colors, width=0.5)
ax1.set_ylabel("Avg Monthly Salary (SGD)", color="#2c6e49")
ax1.tick_params(axis="y", labelcolor="#2c6e49")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax2 = ax1.twinx()
ax2.plot(dept_sum.index, dept_sum["Headcount"], color="#d62728", marker="o", linewidth=2.5, markersize=8)
ax2.set_ylabel("Headcount", color="#d62728"); ax2.tick_params(axis="y", labelcolor="#d62728")
ax1.set_title("Average Salary vs Headcount by Department", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 3 — Performance Score Distribution
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["Performance Score"], bins=14, kde=True, color="#4e9af1", edgecolor="white", ax=ax)
ax.axvline(df["Performance Score"].mean(), color="#e74c3c", linestyle="--", linewidth=2, label=f"Mean={df['Performance Score'].mean():.2f}")
ax.axvline(df["Performance Score"].median(), color="#f39c12", linestyle=":", linewidth=2, label=f"Median={df['Performance Score'].median():.2f}")
ax.set_title("Performance Score Distribution", fontsize=14, fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Chart 4 — Correlation Heatmap
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[["Age","Salary","Years Experience","Performance Score","Monthly Bonus"]].corr().round(3)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5, ax=ax)
ax.set_title("Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 5 — Salary vs Experience Scatter
fig, ax = plt.subplots(figsize=(11, 6))
pal = sns.color_palette("Set2", df["Department"].nunique())
dc  = {d: pal[i] for i,d in enumerate(df["Department"].unique())}
for dept, grp in df.groupby("Department"):
    ax.scatter(grp["Years Experience"], grp["Salary"], label=dept, color=dc[dept], s=70, alpha=0.8)
v = df[["Years Experience","Salary"]].dropna()
p = np.poly1d(np.polyfit(v["Years Experience"], v["Salary"], 1))
x = np.linspace(v["Years Experience"].min(), v["Years Experience"].max(), 100)
ax.plot(x, p(x), "k--", lw=2, alpha=0.7, label="Trend")
ax.set_title("Salary vs Years of Experience", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax.legend(title="Department"); plt.tight_layout(); plt.show()

In [ ]:
# Chart 6 — Gender by Department
gender_dept = df.groupby(["Department","Gender"]).size().unstack(fill_value=0)
ax = gender_dept.plot(kind="bar", stacked=True, figsize=(11,6), color=["#e07b8c","#4e9af1"])
ax.set_title("Gender Representation by Department", fontsize=14, fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 7 — Seniority Donut
order = ["Junior","Mid-Level","Senior","Expert"]
sc = df["Seniority Band"].value_counts().reindex(order, fill_value=0)
fig, ax = plt.subplots(figsize=(7,7))
wedges, texts, auts = ax.pie(sc, labels=sc.index, autopct="%1.1f%%", startangle=140, pctdistance=0.78,
      colors=sns.color_palette("Set2", 4), wedgeprops={"edgecolor":"white","linewidth":2.5})
[t.set_fontweight("bold") for t in auts]
ax.add_artist(plt.Circle((0,0), 0.55, color="white"))
ax.set_title("Seniority Band Distribution", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 8 — Avg Performance by Education
edu_order = ["Diploma","Bachelor's","Master's","PhD"]
edu_perf = df.groupby("Education Level")["Performance Score"].mean().reindex(edu_order).dropna().reset_index()
fig, ax = plt.subplots(figsize=(10,5))
bars = ax.bar(edu_perf["Education Level"], edu_perf["Performance Score"],
              color=sns.color_palette("coolwarm", len(edu_perf)), edgecolor="white", width=0.5)
for bar, val in zip(bars, edu_perf["Performance Score"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03, f"{val:.2f}", ha="center", fontweight="bold")
ax.set_title("Avg Performance Score by Education Level", fontsize=14, fontweight="bold")
ax.set_ylim(0,6); plt.tight_layout(); plt.show()

In [ ]:
# Chart 9 — Executive Dashboard (4-panel)
fig = plt.figure(figsize=(18,12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

ax_a = fig.add_subplot(gs[0,0])
da = df.groupby("Department")["Salary"].mean().sort_values()
bars = ax_a.barh(da.index, da.values, color=sns.color_palette("Set2",len(da)))
[ax_a.text(v+100, bars[i].get_y()+bars[i].get_height()/2, f"${v:,.0f}", va="center", fontsize=9)
 for i,v in enumerate(da.values)]
ax_a.set_title("Avg Salary by Department", fontweight="bold")
ax_a.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f"${x/1000:.0f}k"))

ax_b = fig.add_subplot(gs[0,1])
sns.violinplot(data=df, x="Department", y="Performance Score", palette="Set2", ax=ax_b, inner="quartile", cut=0)
ax_b.set_title("Performance Score — Violin", fontweight="bold")
ax_b.set_xticklabels(ax_b.get_xticklabels(), rotation=20, ha="right")

ax_c = fig.add_subplot(gs[1,0])
sb = df["Salary Band"].value_counts()
ax_c.pie(sb, labels=sb.index, autopct="%1.0f%%", startangle=90,
         colors=sns.color_palette("Set3",len(sb)), wedgeprops={"edgecolor":"white"})
ax_c.set_title("Salary Band Breakdown", fontweight="bold")

ax_d = fig.add_subplot(gs[1,1])
sc_plot = ax_d.scatter(df["Tenure Years"], df["Performance Score"], c=df["Salary"],
                       cmap="YlOrRd", s=70, alpha=0.7)
fig.colorbar(sc_plot, ax=ax_d, label="Salary")
ax_d.set_title("Tenure vs Performance (colour=Salary)", fontweight="bold")
ax_d.set_xlabel("Tenure (Years)"); ax_d.set_ylabel("Performance Score")

fig.suptitle("Employee Analytics — Executive Dashboard", fontsize=17, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

## 8️⃣ AI-Assisted Insights *(Optional)*
> Set your `OPENAI_API_KEY` environment variable to enable live GPT-4o insights.

In [ ]:
import os
try:
    import openai
    api_key = os.environ.get("OPENAI_API_KEY","")
    if not api_key:
        raise ValueError("OPENAI_API_KEY not set")
    client = openai.OpenAI(api_key=api_key)
    prompt = f"""
Employee Dataset Summary:
- Total Employees  : {len(df)}
- Avg Salary       : SGD {df['Salary'].mean():,.0f}
- Avg Perf Score   : {df['Performance Score'].mean():.2f} / 5.0
- Highest Paid Dept: {df.groupby('Department')['Salary'].mean().idxmax()}
- Gender Split     : {df['Gender'].value_counts().to_dict()}
- Seniority Bands  : {df['Seniority Band'].value_counts().to_dict()}

Provide Key Findings, Risks, and Recommendations.
"""
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role":"system","content":"You are a senior data analyst."},
                  {"role":"user","content":prompt}],
        max_tokens=600, temperature=0.4
    )
    print(resp.choices[0].message.content)
except Exception as e:
    print(f"ℹ  AI Insights skipped: {e}")
    print()
    print("Sample prompt that would be sent to GPT-4o:")
    print(f"  - {len(df)} employees analysed")
    print(f"  - Avg Salary: SGD {df['Salary'].mean():,.0f}")
    print(f"  - Top dept by salary: {df.groupby('Department')['Salary'].mean().idxmax()}")
    print("  → Set OPENAI_API_KEY to get live AI narrative insights!")